# `mini-conductor`: Colab H100 / A100 Quickstart

Reproducing & extending Sakana AI's **The Conductor** ([arXiv:2512.04388](https://arxiv.org/abs/2512.04388)) on a single Colab GPU + Claude API.

Every cell is safe to re-run after a disconnect: SFT data cache, checkpoints, the worker cache and eval summaries live on Google Drive and finished work is skipped.

In [ ]:
# 1. Mount Google Drive & check GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
# 2. Point to your uploaded/cloned mini-conductor directory, install, run unit tests
import os
PROJECT_DIR = '/content/drive/MyDrive/mini-conductor'  # Edit if placed elsewhere
EXP_ROOT = '/content/drive/MyDrive/mc_exp'             # checkpoints / cache / eval / figures
os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt
!pytest -q

In [ ]:
# 3. API keys
from getpass import getpass
os.environ['ANTHROPIC_API_KEY'] = getpass('Enter ANTHROPIC_API_KEY: ')
# Optional: gated GPQA-Diamond (otherwise MMLU-Pro STEM is used for the science split)
# os.environ['HF_TOKEN'] = getpass('Enter HF_TOKEN (optional): ')

In [ ]:
# 4. Preflight: ONE minimal request per Claude model referenced by the pool config.
# Must print 3/3 usable before spending anything else.
!python scripts/00_check_api.py --pool configs/pool_claude_hybrid.yaml

In [ ]:
# 5. Datasets: MATH L3-5 / MATH-500 + MBPP + GPQA-Diamond (or MMLU-Pro STEM)
!python scripts/01_build_dataset.py --output_dir data

In [ ]:
# 6. Stage 0: Claude-synthesized, verifier-filtered warm-start trajectories + LoRA SFT
!python scripts/02_gen_sft_data.py --dataset data/train.jsonl --output data/sft_warmup.jsonl --num_samples 300 --cache_db {EXP_ROOT}/sft_cache.sqlite
!python scripts/03_train_sft.py --data data/sft_warmup.jsonl --output_dir {EXP_ROOT}/sft_warmup --epochs 2

In [ ]:
# 7. Short pilot first (10 steps of the base run + small eval) to measure wall-clock and $ per step
#    before committing to the full matrix. Separate root, so it cannot mix with real runs.
!python scripts/06_run_matrix.py --root {EXP_ROOT}_pilot --init_adapter {EXP_ROOT}/sft_warmup --only base --total_steps 10 --save_every 5 --eval_limit 30

In [ ]:
# 8. Full RQ1-RQ4 matrix: 7 training runs -> eval -> figures. Re-run this cell after any disconnect.
!python scripts/06_run_matrix.py --root {EXP_ROOT} --init_adapter {EXP_ROOT}/sft_warmup

In [ ]:
# 9. Figures (drawn only from runs that actually finished)
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{EXP_ROOT}/figures/*.png')):
    print(f); display(Image(f))